In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
PROJECT_DIR = '/content/drive/MyDrive/cloud-feature-bank'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/data', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/indexes', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/features', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/metadata', exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Contents:", os.listdir(PROJECT_DIR))

Project directory: /content/drive/MyDrive/cloud-feature-bank
Contents: ['data', 'indexes', 'features', 'metadata']


In [3]:
!pip install -q faiss-cpu transformers torch torchvision pillow numpy
print("Done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 67.6 MB/s eta 0:00:00
Done


In [4]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("PyTorch version:", torch.__version__)

CUDA available: True
Device: Tesla T4
PyTorch version: 2.10.0+cu128


In [5]:
import torch
from transformers import AutoImageProcessor, AutoModel

MODEL_NAME = "facebook/dinov2-small"  # ViT-S/14, 384-dim features
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device).eval()

# Quick test: get feature dim
dummy = torch.randn(1, 3, 224, 224).to(device)
with torch.no_grad():
    out = model(dummy)
    feature_dim = out.last_hidden_state[:, 0, :].shape[-1]

print(f"Model loaded: {MODEL_NAME}")
print(f"Feature dimension: {feature_dim}")
print(f"Device: {device}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

The image processor of type `BitImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/88.2M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Model loaded: facebook/dinov2-small
Feature dimension: 384
Device: cuda


In [6]:
import torchvision
import torchvision.transforms as transforms

# DINOv2 expects images at 224x224 with ImageNet normalization
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# Download CIFAR-100 train set
train_dataset = torchvision.datasets.CIFAR100(
    root=f'{PROJECT_DIR}/data',
    train=True,
    download=True,
    transform=transform
)

# Download CIFAR-100 test set
test_dataset = torchvision.datasets.CIFAR100(
    root=f'{PROJECT_DIR}/data',
    train=False,
    download=True,
    transform=transform
)

print(f"Train images: {len(train_dataset)}")
print(f"Test images: {len(test_dataset)}")
print(f"Number of classes: {len(train_dataset.classes)}")
print(f"First 5 classes: {train_dataset.classes[:5]}")

100%|██████████| 169M/169M [00:04<00:00, 39.7MB/s]


Train images: 50000
Test images: 10000
Number of classes: 100
First 5 classes: ['apple', 'aquarium_fish', 'baby', 'bear', 'beaver']


In [7]:
from torch.utils.data import DataLoader, Subset
import numpy as np
import time

# Just take first 10 images
small_subset = Subset(train_dataset, range(10))
loader = DataLoader(small_subset, batch_size=10, shuffle=False, num_workers=2)

start = time.time()
with torch.no_grad():
    for images, labels in loader:
        images = images.to(device)
        outputs = model(images)
        # Use CLS token (global feature) — first token of last_hidden_state
        features = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        print(f"Batch shape: {features.shape}")
        print(f"First feature vector (first 5 values): {features[0][:5]}")
        print(f"Labels: {labels.tolist()}")

print(f"\nTime for 10 images: {time.time()-start:.2f}s")

Batch shape: (10, 384)
First feature vector (first 5 values): [ 2.7868145e+00  1.0848254e+00  6.6756806e+00 -2.6508837e-03
  1.9303287e+00]
Labels: [19, 29, 0, 11, 1, 86, 90, 28, 23, 31]

Time for 10 images: 0.32s


In [8]:
from tqdm import tqdm

BATCH_SIZE = 128
loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

all_features = []
all_labels = []

start = time.time()
with torch.no_grad():
    for images, labels in tqdm(loader, desc="Extracting features"):
        images = images.to(device)
        outputs = model(images)
        # Take CLS token features
        features = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        all_features.append(features)
        all_labels.append(labels.numpy())

# Concatenate all batches
all_features = np.concatenate(all_features, axis=0).astype('float32')
all_labels = np.concatenate(all_labels, axis=0).astype('int64')

print(f"\nTotal features shape: {all_features.shape}")
print(f"Total labels shape: {all_labels.shape}")
print(f"Time: {(time.time()-start)/60:.1f} min")

Extracting features: 100%|██████████| 391/391 [04:04<00:00,  1.60it/s]


Total features shape: (50000, 384)
Total labels shape: (50000,)
Time: 4.1 min


In [9]:
features_path = f'{PROJECT_DIR}/features/cifar100_vits14_train_features.npy'
labels_path = f'{PROJECT_DIR}/metadata/cifar100_vits14_train_labels.npy'

np.save(features_path, all_features)
np.save(labels_path, all_labels)

# Also save class names for later
import json
class_names = train_dataset.classes
metadata = {
    "model": MODEL_NAME,
    "feature_dim": int(all_features.shape[1]),
    "num_train_images": int(len(all_features)),
    "num_classes": len(class_names),
    "class_names": class_names,
    "dataset": "CIFAR-100",
    "split": "train"
}
with open(f'{PROJECT_DIR}/metadata/cifar100_vits14_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Saved features to: {features_path}")
print(f"Saved labels to: {labels_path}")
print(f"Saved metadata.json")
print(f"Features file size: {os.path.getsize(features_path) / 1024 / 1024:.1f} MB")

Saved features to: /content/drive/MyDrive/cloud-feature-bank/features/cifar100_vits14_train_features.npy
Saved labels to: /content/drive/MyDrive/cloud-feature-bank/metadata/cifar100_vits14_train_labels.npy
Saved metadata.json
Features file size: 73.2 MB


In [10]:
import faiss

# Normalize for cosine similarity (inner product on normalized = cosine)
features_normalized = all_features.copy()
faiss.normalize_L2(features_normalized)

# Build index — IndexFlatIP for exact search (small dataset, exact is fine)
index = faiss.IndexFlatIP(features_normalized.shape[1])
index.add(features_normalized)

print(f"Index built with {index.ntotal} vectors")
print(f"Feature dimension: {index.d}")

# Save index
index_path = f'{PROJECT_DIR}/indexes/cifar100_vits14.faiss'
faiss.write_index(index, index_path)
print(f"Index saved to: {index_path}")
print(f"Index file size: {os.path.getsize(index_path) / 1024 / 1024:.1f} MB")

Index built with 50000 vectors
Feature dimension: 384
Index saved to: /content/drive/MyDrive/cloud-feature-bank/indexes/cifar100_vits14.faiss
Index file size: 73.2 MB


In [11]:
# Extract test features (smaller — only 10K images)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

test_features = []
test_labels = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Extracting test features"):
        images = images.to(device)
        outputs = model(images)
        features = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        test_features.append(features)
        test_labels.append(labels.numpy())

test_features = np.concatenate(test_features, axis=0).astype('float32')
test_labels = np.concatenate(test_labels, axis=0).astype('int64')

# Normalize test features (same as train)
test_features_normalized = test_features.copy()
faiss.normalize_L2(test_features_normalized)

# Search for top-K neighbors
K = 20
distances, indices = index.search(test_features_normalized, K)

# Predict via majority vote of K neighbors' labels
predictions = []
for neighbor_ids in indices:
    neighbor_labels = all_labels[neighbor_ids]
    # Majority vote
    pred = np.bincount(neighbor_labels).argmax()
    predictions.append(pred)
predictions = np.array(predictions)

accuracy = (predictions == test_labels).mean()
print(f"\nkNN classification accuracy on CIFAR-100 test set (K={K}): {accuracy*100:.2f}%")

Extracting test features: 100%|██████████| 79/79 [00:51<00:00,  1.53it/s]



kNN classification accuracy on CIFAR-100 test set (K=20): 80.43%


In [12]:
np.save(f'{PROJECT_DIR}/features/cifar100_vits14_test_features.npy', test_features)
np.save(f'{PROJECT_DIR}/metadata/cifar100_vits14_test_labels.npy', test_labels)
print("Test features saved")

Test features saved
